# Kiểm lại một bản phân tích interpretability

Một đồng nghiệp gửi bạn notebook này: bốn thí nghiệm trên GPT-2 small và bốn kết luận đã viết sẵn trong báo cáo nội bộ. Notebook
chạy hết, không lỗi, không cảnh báo. Trước khi kết luận đi tiếp, bạn là người review.

**Cách làm bài**
- Các cell định nghĩa hàm là pipeline của đồng nghiệp; test chấm các hàm đó.
- Thêm cell `explore` tuỳ ý để kiểm. Sửa ngay trong các cell hàm; không đổi tên, chữ ký hay docstring.
- Ghi điều tra vào `BUGS.md`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/08-interpretability/exercises/advanced/02-debug-interpretability -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải IWSLT15 nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
IWSLT = ROOT / "data/raw/iwslt15-en-vi"
if not (IWSLT.is_dir() and any(IWSLT.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("iwslt15-en-vi")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Pipeline

Bốn hàm phân tích và một SAE TopK. Model là lớp `GPT` của module 05 (pre-norm, `ln_f`, `lm_head` dùng chung `tok_emb`).

In [ ]:
def cache_streams(model, idx):
    """Chạy model một lần, trả (logits, streams): streams[l] là residual stream ra khỏi Block l, shape (B, T, C)."""
    streams, handles = {}, []
    for l, block in enumerate(model.blocks):
        handles.append(block.mlp.register_forward_hook(lambda m, args, out, l=l: streams.__setitem__(l, out.detach().clone())))
    try:
        with torch.no_grad():
            logits, _ = model(idx)
    finally:
        for h in handles:
            h.remove()
    return logits, streams

In [ ]:
def lens_report(model, idx):
    """Logit lens trên mọi tầng: list dict {"layer", "ce", "top1_agree"} (CE với token kế tiếp, tỉ lệ trùng argmax của model)."""
    logits, streams = cache_streams(model, idx)
    targets, final_pred = idx[:, 1:], logits[:, :-1].argmax(-1)
    rows = []
    for l in sorted(streams):
        with torch.no_grad():
            lg = model.lm_head(model.ln_f(streams[l]))[:, :-1]
        rows.append({"layer": l, "ce": F.cross_entropy(lg.transpose(1, 2), targets).item(),
                     "top1_agree": (lg.argmax(-1) == final_pred).float().mean().item()})
    return rows

In [ ]:
def probe_with_controls(X, y, token_ids, groups, seed=0, weight_decay=1e-3):
    """Probe tuyến tính nhị phân có đối chứng. Chia theo nhóm (25% nhóm vào test), chuẩn hoá theo train, train probe cho nhãn
    thật và cho control task (Hewitt & Liang: mỗi loại token một nhãn ngẫu nhiên theo phân phối nhãn). Trả dict
    {"task", "control", "selectivity", "majority"} (accuracy trên test)."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    test_groups = uniq[rng.permutation(len(uniq))[: int(round(0.25 * len(uniq)))]]
    te = np.isin(groups, test_groups)
    tr = ~te
    mean, std = X[tr].mean(0), X[tr].std(0) + 1e-6
    Xs = (X - mean) / std
    values, counts = np.unique(y, return_counts=True)
    control = rng.choice(values, size=len(token_ids), p=counts / counts.sum())

    def fit_eval(labels):
        torch.manual_seed(seed)
        Xt, yt = torch.tensor(Xs[tr], dtype=torch.float32), torch.tensor(labels[tr])
        probe = torch.nn.Linear(X.shape[1], 2)
        opt = torch.optim.LBFGS(probe.parameters(), max_iter=200, line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(probe(Xt), yt) + weight_decay * probe.weight.pow(2).sum()
            loss.backward()
            return loss

        opt.step(closure)
        with torch.no_grad():
            pred = probe(torch.tensor(Xs[te], dtype=torch.float32)).argmax(-1).numpy()
        return float((pred == labels[te]).mean())

    task, ctrl = fit_eval(y), fit_eval(control)
    majority = float((y[te] == values[counts.argmax()]).mean())
    return {"task": task, "control": ctrl, "selectivity": task - ctrl, "majority": majority}

In [ ]:
def patch_resid(model, idx, source_streams, layer, pos):
    """Chạy `model(idx)` nhưng thay residual stream **đi vào** Block `layer` ở vị trí `pos` bằng giá trị tương ứng của run kia.
    `source_streams` là dict {"embed": stream đi vào Block 0, l: stream ra khỏi Block l} của run kia. Trả logits."""
    source = source_streams["embed"] if layer == 0 else source_streams[layer - 1]

    def hook(module, args):
        x = args[0].clone()
        x[:, pos] = source[:, pos - 1]
        return (x,)

    handle = model.blocks[layer].register_forward_pre_hook(hook)
    try:
        with torch.no_grad():
            return model(idx)[0]
    finally:
        handle.remove()

In [ ]:
@torch.no_grad()
def sae_report(sae, acts):
    """Đánh giá SAE trên `acts` (N, d): dict {"fvu", "l0"}. FVU = phần phương sai không giải thích được (1 - R^2)."""
    x_hat, f = sae(acts)
    fvu = (acts - x_hat).pow(2).sum() / acts.pow(2).sum()
    return {"fvu": fvu.item(), "l0": (f > 0).float().sum(-1).mean().item()}

In [ ]:
class TopKSAE(nn.Module):
    """SAE TopK đơn giản: encode = TopK(ReLU((x - b_dec) W_enc + b_enc)), decode = f W_dec + b_dec."""

    def __init__(self, d_in, d_sae, k, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        W = torch.randn(d_sae, d_in, generator=g)
        W = W / W.norm(dim=1, keepdim=True)
        self.W_dec, self.W_enc = nn.Parameter(W.clone()), nn.Parameter(W.T.clone())
        self.b_enc, self.b_dec = nn.Parameter(torch.zeros(d_sae)), nn.Parameter(torch.zeros(d_in))
        self.k = k

    def forward(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        v, i = pre.topk(self.k, dim=-1)
        f = torch.zeros_like(pre).scatter(-1, i, F.relu(v))
        return f @ self.W_dec + self.b_dec, f

## Thí nghiệm của đồng nghiệp

Các cell dưới không được chấm.

In [ ]:
# explore: nạp lớp GPT (module 05) và GPT-2 small; văn bản TED
import html
import importlib.util
import json
import re

from transformers import AutoTokenizer, GPT2LMHeadModel

torch.set_num_threads(4)
M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture/solutions/basic/01-gpt-mini/gpt.py"
spec = importlib.util.spec_from_file_location("gpt_mini", M05)
gpt_mini = importlib.util.module_from_spec(spec)
sys.modules["gpt_mini"] = gpt_mini
spec.loader.exec_module(gpt_mini)


def notebook_namespace(path):
    """Chạy các cell code không gắn tag explore của một notebook, trả namespace (giống cách test nạp notebook)."""
    ns = {}
    for cell in json.loads(Path(path).read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code" and "explore" not in cell["metadata"].get("tags", []):
            exec("".join(cell["source"]), ns)
    return ns


def load_gpt2(gpt_mini):
    """GPT-2 small trong lớp GPT bằng gpt2_into_gpt_mini của bạn (bài Intermediate 01); lệch logits so với Hugging Face quá 1e-3
    hoặc lỗi bất kỳ thì dùng lời giải và in cảnh báo."""
    I01 = ROOT / "tracks/ai-engineer/modules/08-interpretability/{}/intermediate/01-gpt2-heads-lens-probe/{}inside_gpt2.ipynb"
    hf = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
    probe_ids = torch.tensor([[50256, 464, 412, 733, 417, 8765, 318, 5140, 287, 262, 1748, 286]])
    with torch.no_grad():
        ref = hf(probe_ids).logits
    for d, sub in [("exercises", "starter/"), ("solutions", "")]:
        try:
            mdl = notebook_namespace(str(I01).format(d, sub))["gpt2_into_gpt_mini"](gpt_mini, hf)
            with torch.no_grad():
                diff = (mdl(probe_ids)[0] - ref).abs().max().item()
            if diff <= 1e-3:
                print(f"gpt2_into_gpt_mini từ {d} (bài Intermediate 01), lệch logits {diff:.1e}")
                return mdl.eval()
            print(f"cảnh báo: gpt2_into_gpt_mini ở {d} lệch logits {diff:.1e} > 1e-3, chuyển sang lời giải")
        except NotImplementedError:
            continue
        except Exception as e:
            print(f"cảnh báo: gpt2_into_gpt_mini ở {d} lỗi ({type(e).__name__}: {e}), chuyển sang lời giải")
    raise RuntimeError("không nạp được GPT-2")


tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = load_gpt2(gpt_mini)
BOS = tok.bos_token_id


def detok(s):
    s = html.unescape(s)
    s = re.sub(r" ([.,!?;:%)\]])", r"\1", s)
    s = re.sub(r" (n't|'s|'re|'ve|'m|'ll|'d)\b", r"\1", s)
    return re.sub(r'" (.*?) "', r'"\1"', s)


ted = [l for s in ["tst2012", "tst2013"] for l in (IWSLT / f"{s}.en").read_text(encoding="utf-8").splitlines()]
ids = tok.encode(" ".join(detok(s) for s in ted))
text_idx = torch.cat([torch.full((80, 1), BOS), torch.tensor(ids[: 80 * 127]).view(80, 127)], dim=1)
print(text_idx.shape)

In [ ]:
# explore: thí nghiệm 1, logit lens trên 20 ngữ cảnh
for row in lens_report(model, text_idx[:20]):
    print(f"tầng {row['layer']:2d}: CE {row['ce']:6.2f}  top1 trùng model {row['top1_agree']:.3f}")

In [ ]:
# explore: thí nghiệm 2, probe "token đã xuất hiện trong ngữ cảnh chưa" ở stream ra khỏi tầng 1
labels = np.zeros(text_idx.shape, dtype=np.int64)
for b in range(len(text_idx)):
    seen = set()
    for t in range(1, text_idx.shape[1]):
        labels[b, t] = int(text_idx[b, t]) in seen
        seen.add(int(text_idx[b, t]))
_, streams = cache_streams(model, text_idx)
X = streams[1][:, 1:].reshape(-1, 768).numpy()
y = labels[:, 1:].reshape(-1)
toks = text_idx[:, 1:].numpy().reshape(-1)
groups = np.repeat(np.arange(len(text_idx)), text_idx.shape[1] - 1)
print(probe_with_controls(X, y, toks, groups))

In [ ]:
# explore: thí nghiệm 3, patching residual trên 16 cặp prompt IOI (clean/corrupted khác nhau ở vị trí 10)
import random

NAMES = ["Mary", "John", "Tom", "James", "Dan", "Sid", "Martin", "Amy", "Kate", "Paul", "Mark", "Anna"]
rng = random.Random(1)
clean_txt, corrupt_txt, io, s = [], [], [], []
for i in range(16):
    a, b = rng.sample(NAMES, 2)
    subj, ind = (b, a) if i % 2 == 0 else (a, b)
    clean_txt.append(f"When {a} and {b} went to the store, {subj} gave a drink to")
    corrupt_txt.append(f"When {a} and {b} went to the store, {ind} gave a drink to")
    io.append(tok.encode(" " + ind)[0]); s.append(tok.encode(" " + subj)[0])
clean = torch.tensor([[BOS] + tok.encode(p) for p in clean_txt])
corrupt = torch.tensor([[BOS] + tok.encode(p) for p in corrupt_txt])
io, s = torch.tensor(io), torch.tensor(s)
metric = lambda lg: (lg[torch.arange(16), -1, io] - lg[torch.arange(16), -1, s]).mean().item()
clean_logits, clean_streams = cache_streams(model, clean)
store = {}
h = model.blocks[0].register_forward_pre_hook(lambda m, a: store.__setitem__("embed", a[0].detach().clone()))
with torch.no_grad():
    model(clean)
h.remove()
clean_streams["embed"] = store["embed"]
corrupt_logits, _ = cache_streams(model, corrupt)
LC, LX = metric(clean_logits), metric(corrupt_logits)
print(f"logit diff clean {LC:.2f}, corrupted {LX:.2f}")
table = torch.zeros(4, clean.shape[1])
for i, layer in enumerate([0, 4, 8, 11]):
    for pos in range(clean.shape[1]):
        table[i, pos] = (metric(patch_resid(model, corrupt, clean_streams, layer, pos)) - LX) / (LC - LX)
torch.set_printoptions(precision=2, sci_mode=False, linewidth=160)
print(tok.convert_ids_to_tokens(clean[0].tolist()))
print(table)

In [ ]:
# explore: thí nghiệm 4, SAE TopK trên dữ liệu có 30 đặc trưng thưa trong 10 chiều
g = torch.Generator().manual_seed(0)
dirs = torch.randn(30, 10, generator=g)
dirs /= dirs.norm(dim=1, keepdim=True)
data = ((torch.rand(20000, 30, generator=g) < 0.05).float() * (0.5 + torch.rand(20000, 30, generator=g))) @ dirs
data = data + 3.0
sae = TopKSAE(10, 60, k=3)
with torch.no_grad():
    sae.b_dec.copy_(data.mean(0))
opt = torch.optim.Adam(sae.parameters(), lr=3e-3)
for step in range(300):
    x = data[torch.randint(len(data), (256,), generator=g)]
    loss = (sae(x)[0] - x).pow(2).sum(-1).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        sae.W_dec /= sae.W_dec.norm(dim=1, keepdim=True)
print(sae_report(sae, data))

## Kết luận đồng nghiệp đã viết vào báo cáo

1. "Logit lens cho thấy GPT-2 small gần như chưa có dự đoán nào trước tầng cuối: lens ở các tầng giữa cho cross-entropy rất cao và
   gần như không bao giờ trùng top-1 với model."
2. "Thông tin 'token đã xuất hiện' được mã hoá rất chọn lọc ở tầng 1: selectivity gần bằng accuracy của probe trừ majority
   baseline, tức probe không thể học thuộc theo loại token."
3. "Trong bài IOI, thông tin từ vị trí S2 (vị trí 10) không đến thẳng tầng 0: patch tầng 0 ở vị trí 10 gần như không khôi phục gì,
   hiệu ứng rải rác ở các vị trí khác."
4. "SAE TopK chỉ sau 300 bước đã tái dựng gần như hoàn hảo dữ liệu (FVU dưới 0.01), nên ta có thể tin các latent của nó."

Kiểm từng kết luận. Kết luận nào sai, vì sao, sửa ở đâu; sau khi sửa, kết luận đúng là gì?